# ⭐ Project · The Nimbus Briefing Assistant

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama and this section's
> `mcp_server.py`. The full run makes a few dozen model calls; on a local model
> expect several minutes.

**Nimbus Robotics** (the fictional company in `data/kb/`) is rolling out 180
delivery robots to its customer Albers Markt, and the rollout is slipping. On the
last steering call, Albers Markt asked for *"a written briefing with a realistic
date per store group"*. The facts are spread across six documents (a battery
shortage, a delayed software release, a permit, customer feedback), so no single
lookup answers it.

You'll build the assistant that writes that briefing with the Agents SDK. It's
the same project as in the LangGraph section, built the SDK way: **agents
orchestrated by plain Python**, with the durability and approval pieces added
where the SDK expects you to add them.

**What you'll practise**

| Feature | From notebook |
|---|---|
| MCP tools, filtered per agent, with approval on the write tool | 11 |
| A planner with structured output (`output_type`) | 03 |
| Parallel research agents with `asyncio.gather` and a concurrency cap | 05 |
| A writer + reviewer loop with a cap, exact checks in code | 05 |
| Step checkpoints so a crash doesn't repeat finished work | 09 |
| Approval of the save, paused to JSON and resumed | 08 |
| Verifying the side effect in the system of record | 06, 11 |
| Retries for leaked tool calls (`run_agent`) and token accounting | 11, 12 |

## 1 · The design

```
                   ┌──────────── asyncio.gather (max 3 at once) ────────────┐
 request ──► PLAN ─┼─► research(q1) ─┐                                       │
  (Planner agent,  ├─► research(q2) ─┼──► WRITE ──► REVIEW ──pass──► PUBLISH (save_report needs approval)
   output_type)    └─► research(qN) ─┘      ▲          │                 │ run pauses → state saved as JSON
                     (Researcher agents,     └─revise──┘ fail, < 2 revisions   │ human approves → resume
                      read-only MCP tools)      fail ≥ 2 → PUBLISH, flagged   ▼
                                                                         verify the file exists
   every step's result is checkpointed to disk → a crash re-runs only unfinished steps
```

Design decisions, and why:

- **The flow is code** (notebook 05). Plan → research → write → review → publish is
  fixed, so Python sequences it. Agents do the judgement work *inside* each step.
- **Least privilege** (notebook 11). Researchers connect to the MCP server through a
  read-only filter. Only the publisher sees `save_report`, and it requires approval.
- **Durability is ours to add** (notebook 09). The SDK doesn't checkpoint between
  our `await`s, so a small `step()` helper saves each step's result to disk and
  reuses it after a crash.
- **Trust, but verify** (notebooks 06, 11). After publishing, we check the reports
  folder, not the agent's confirmation.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

import asyncio, json, pathlib, shutil, sys, re
from pydantic import BaseModel, Field
from agents import Agent, Runner, RunState
from agents.mcp import MCPServerStdio, create_static_tool_filter
from common import get_model, run_agent, SECTION_DIR

model = get_model()
SERVER = {'command': sys.executable, 'args': [str(SECTION_DIR / 'mcp_server.py')]}
REPORTS = SECTION_DIR / 'data' / 'reports'

## 2 · Connections to the MCP server

Two connections to the same server, with different privileges. Because they
must stay open across several cells, we use `connect()` / `cleanup()` instead
of an `async with` block (we close them at the end of the notebook).

One MCP detail: when a tool returns a **list**, it arrives as **one content block
per item**. Our first version read only `content[0]` and saw a single document.

In [ ]:
kb_read = MCPServerStdio(name='kb-read', params=SERVER, cache_tools_list=True,
                         tool_filter=create_static_tool_filter(
                             allowed_tool_names=['list_documents', 'read_document', 'search_documents']))
kb_publish = MCPServerStdio(name='kb-publish', params=SERVER,
                            tool_filter=create_static_tool_filter(allowed_tool_names=['save_report']),
                            require_approval={'always': {'tool_names': ['save_report']}})
await kb_read.connect()
await kb_publish.connect()

print('researchers see:', [t.name for t in await kb_read.list_tools()])
print('publisher sees :', [t.name for t in await kb_publish.list_tools()])

listing = await kb_read.call_tool('list_documents', {})
docs = [json.loads(block.text) for block in listing.content]      # a list comes back as one block per item
DOC_NAMES = [d['name'] for d in docs]
KB_INDEX = '\n'.join(DOC_NAMES)
print('\ndocuments:', DOC_NAMES)

## 3 · The agents

Five small agents, each with one job. They're built in a function that takes
the model, so they could be tested with a scripted fake model (notebook 12).

In [ ]:
class Plan(BaseModel):
    questions: list[str] = Field(description='2 to 4 specific research questions')

class Verdict(BaseModel):
    supported: bool
    problems: str


def build_agents(model):
    return {
        'planner': Agent(name='Planner', model=model, output_type=Plan, instructions=(
            'You plan research for a customer briefing. Given the request and the list of available '
            'documents, write 2-4 specific research questions whose answers the briefing needs. '
            'Each question is a SEPARATE item.')),
        'researcher': Agent(name='Researcher', model=model, mcp_servers=[kb_read], instructions=(
            'You research Nimbus Robotics using ONLY the knowledge-base tools. Search, then READ the '
            'relevant documents. Answer in at most 4 sentences with concrete numbers and dates. '
            'End with "Sources: <document names>".')),
        'writer': Agent(name='Writer', model=model, instructions=(
            'Write a briefing for Albers Markt in markdown with exactly these sections: '
            '## Summary, ## Go-live dates per store group, ## Risks, ## Sources. Use ONLY facts from '
            'the findings; never invent numbers or dates. In Sources, list the document names used.')),
        'reviewer': Agent(name='Reviewer', model=model, output_type=Verdict, instructions=(
            'Check the briefing against the findings. supported = true only if every claim is backed by '
            'the findings. In problems, list unsupported or wrong claims, or write "none".')),
        'publisher': Agent(name='Publisher', model=model, mcp_servers=[kb_publish], instructions=(
            'Save the briefing with the save_report tool, titled "Albers Markt rollout briefing", '
            'with the briefing text exactly as given. Then confirm.')),
    }

agents = build_agents(model)
print({k: a.name for k, a in agents.items()})

## 4 · Durable steps, and token accounting

`step(name, work)` is the checkpoint pattern from notebook 09: if the step's
result is already on disk, reuse it; otherwise run it and save the result
**before** moving on. Each step returns plain JSON data (strings, lists, dicts),
so it's easy to save.

In [ ]:
CKPT = pathlib.Path('checkpoints/project')
USAGE: dict[str, int] = {}
RAN: list[str] = []                        # which steps actually executed in this process


async def step(name: str, work):
    path = CKPT / f'{name}.json'
    if path.exists():
        print(f'  ↺ {name:<14} (from checkpoint)')
        return json.loads(path.read_text())
    print(f'  ▶ {name}')
    RAN.append(name)
    value = await work()
    path.write_text(json.dumps(value))     # durable before the next step starts
    return value


def count(label: str, result):
    USAGE[label] = USAGE.get(label, 0) + result.context_wrapper.usage.total_tokens

## 5 · The workflow, as plain Python

Read `briefing_pipeline` top to bottom: it *is* the design diagram.

Notice the two **gates** (notebook 05). While building this project, the planner
once returned an **empty** list of questions. With no questions, no researcher
ran, and the writer happily produced a confident briefing about a "retail
management system" citing documents that don't exist. The reviewer flagged it,
but only after tokens were spent. Now an empty plan falls back to researching
the request itself, and empty findings stop the pipeline before any writing.
`FAIL_WRITE_ONCE` lets us simulate a crash in the writer.

In [ ]:
MAX_REVISIONS = 2
MAX_PARALLEL = 3
FAIL_WRITE_ONCE = {'armed': False}
REQUIRED_SECTIONS = ('## Summary', '## Go-live dates', '## Risks', '## Sources')


async def plan(request):
    result = await run_agent(agents['planner'], f'Request: {request}\n\nAvailable documents:\n{KB_INDEX}')
    count('plan', result)
    questions = [q for q in result.final_output.questions if q.strip()][:4]
    return questions or [request]                                   # gate: never research nothing


async def research(request, questions):
    gate = asyncio.Semaphore(MAX_PARALLEL)                        # at most 3 researchers at once
    async def one(q):
        async with gate:
            result = await run_agent(agents['researcher'], f'Context: {request}\nQuestion: {q}')
            count('research', result)
            return f'### {q}\n{result.final_output}'
    return list(await asyncio.gather(*(one(q) for q in questions)))


async def write(findings, previous='', feedback=''):
    if FAIL_WRITE_ONCE['armed']:
        FAIL_WRITE_ONCE['armed'] = False
        raise RuntimeError('simulated crash: the writer process died mid-run')
    prompt = 'Findings:\n\n' + '\n\n'.join(findings)
    if feedback:
        prompt += f'\n\nPrevious draft:\n{previous}\n\nFix these problems:\n{feedback}'
    result = await run_agent(agents['writer'], prompt)
    count('write', result)
    return result.final_output


async def review(findings, draft):
    problems = [f'missing section "{s}"' for s in REQUIRED_SECTIONS if s.lower() not in draft.lower()]
    if not any(d in draft for d in DOC_NAMES):
        problems.append(f'Sources must name knowledge-base documents, e.g. {DOC_NAMES[2]}')
    if not problems:                                              # exact checks first, then judgement
        result = await Runner.run(agents['reviewer'],
                                  'Findings:\n' + '\n\n'.join(findings) + f'\n\nBriefing:\n{draft}')
        count('review', result)
        if not result.final_output.supported:
            problems.append(result.final_output.problems)
    return problems


async def briefing_pipeline(request):
    questions = await step('plan', lambda: plan(request))
    print('    plan:', *[f'\n      • {q[:110]}' for q in questions])
    findings = await step('research', lambda: research(request, questions))
    if not findings:                                                # gate: no facts, no briefing
        raise RuntimeError('research produced no findings; refusing to write a briefing from nothing')

    draft, problems = '', []
    for revision in range(1, MAX_REVISIONS + 1):
        draft = await step(f'write_v{revision}', lambda: write(findings, draft, '; '.join(problems)))
        problems = await step(f'review_v{revision}', lambda: review(findings, draft))
        print(f'    review v{revision}:', 'PASS' if not problems else '; '.join(problems)[:150])
        if not problems:
            break
    status = 'passed review' if not problems else f'⚠️ FLAGGED after {MAX_REVISIONS} revisions: ' + '; '.join(problems)
    return {'questions': questions, 'findings': findings, 'draft': draft, 'review': status}

## 6 · Run it, with a crash in the middle

We arm `FAIL_WRITE_ONCE`, so the writer fails the first time, after planning
and all research have finished and been checkpointed. Then we simply **call the
pipeline again**, as a restarted process would. Watch which steps run.

In [ ]:
REQUEST = ('Albers Markt asked for a written briefing on the Courier C2 rollout to their 12 stores, '
           'with a realistic go-live date per store group and the main risks.')

shutil.rmtree(CKPT, ignore_errors=True)
CKPT.mkdir(parents=True)
USAGE.clear(); RAN.clear()

FAIL_WRITE_ONCE['armed'] = True
print('attempt 1:')
try:
    await briefing_pipeline(REQUEST)
except RuntimeError as err:
    print('\n💥', err)
print('checkpoints on disk:', sorted(p.stem for p in CKPT.glob('*.json')))

In [ ]:
print('attempt 2 (after the "restart"):')
RAN.clear()
briefing = await briefing_pipeline(REQUEST)
print('\nsteps executed in attempt 2:', RAN)

Planning and research came **from checkpoints**: no planner call, no research
agents, no tokens. Only the writer (which crashed) and the steps after it ran.
That's exactly what the SDK doesn't do for you, added with fifteen lines of code.

## 7 · Publish, with a human approval

The publisher agent calls `save_report`, which requires approval, so the run
**pauses**. We save the paused run to JSON (it could wait days), show the human
the draft and the review status, and resume with their decision.

In [ ]:
result = await run_agent(agents['publisher'], briefing['draft'])
pending = result.interruptions[0]
(CKPT / 'publish_pending.json').write_text(json.dumps(result.to_state().to_json()))

print('PAUSED: waiting for approval of', pending.tool_name)
print('REVIEW STATUS:', briefing['review'])
print('─' * 100)
print(briefing['draft'])

Read the draft critically, as the human in the loop. Are the dates consistent
with the documents? (The battery incident, the 4.2 release estimate and the
customer's "8 of 12 by 30 September" condition all matter.)

**What our test run showed.** The draft passed the automated review, yet it
added a year ("30 September **2024**") that appears in no document, and listed
"potential permit delays in Utrecht" although `05_permits_and_regulation` says the
permit was granted on 8 August. A model reviewer checks what it's asked, and
misses what it isn't. That's why the human step exists, and why exercise 1 adds a
stricter code check. Your run will differ, so read your draft the same way.

Then approve, or reject with a reason. We approve here. Try `reject` in your own run.

Note that we reload the state **from the JSON file**, as a separate reviewer
process would:

In [ ]:
before = set(REPORTS.glob('*.md')) if REPORTS.exists() else set()

state = await RunState.from_json(agents['publisher'], json.loads((CKPT / 'publish_pending.json').read_text()))
state.approve(state.get_interruptions()[0])
published = await Runner.run(agents['publisher'], state)
count('publish', published)

new_files = sorted(set(REPORTS.glob('*.md')) - before)
print('publisher says :', published.final_output[:150])
print('verified on disk:', [f.relative_to(SECTION_DIR).as_posix() for f in new_files] or 'NOTHING SAVED')

## 8 · What did it cost?

In [ ]:
print('tokens per step (both attempts):', USAGE)
print('total:', sum(USAGE.values()))
await kb_read.cleanup()
await kb_publish.cleanup()

## 9 · Compare with the LangGraph version

If you did the LangGraph section, you built the same assistant there. Here's
how the same design decisions were expressed:

| Design decision | LangGraph | Agents SDK |
|---|---|---|
| The fixed flow | a graph: nodes, edges, a conditional edge | an `async` Python function |
| Parallel research | `Send` + a reducer | `asyncio.gather` + a `Semaphore` |
| Research worker | `create_agent` inside a node | an `Agent` with MCP tools |
| Crash recovery | automatic checkpoints (`durability='sync'`) | our `step()` helper |
| Human approval | `interrupt()` in a node | `require_approval` on the MCP tool + `RunState` JSON |
| Refine loop guard | revision counter in state + router | a `for` loop with a cap |
| Streaming progress | `stream_mode=['updates','custom']` | `print`s in our steps (or hooks / `run_streamed`) |

Neither is "better". The SDK version is shorter and reads like ordinary Python,
but you wrote the durability yourself and it only covers the steps you wrapped.
The LangGraph version gets per-step checkpoints, time travel and pausing
anywhere from the framework, at the cost of learning and designing a graph.

## 10 · Exercises

1. **Grounding check** (05, 12): add a code check to `review` that every date in the
   draft appears in some finding.
2. **Reject path** (08): reject the publish with a reason, and make the pipeline
   return to `write` with that reason as feedback.
3. **Streaming** (10): show research progress live with `run_streamed` and item events.
4. **Tests** (12): build the agents with a scripted fake model and test that a
   failing review loops at most `MAX_REVISIONS` times, then publishes flagged.
5. **Handoffs** (06): add a "customer tone" editor the writer can hand off to.

## 11 · Check yourself

<details><summary><b>1.</b> Why didn't attempt 2 call the planner or the researchers?</summary>

Their results had been saved by `step()` before the crash. On the rerun, `step()` found the checkpoint files and returned the saved results instead of running the work again.
</details>

<details><summary><b>2.</b> Why do the researchers and the publisher use different MCP connections?</summary>

Least privilege: researchers only need read tools, so their connection's filter hides `save_report`. The publisher's connection exposes only `save_report`, and it requires approval.
</details>

<details><summary><b>3.</b> What makes the publish step safe to leave waiting for days?</summary>

The paused run is saved as JSON (`to_state().to_json()`), so no process has to stay alive. Any process with the same agent definitions can load it, record the decision and resume.
</details>

<details><summary><b>4.</b> Why check the reports folder instead of trusting the publisher's reply?</summary>

An agent's words aren't evidence that an action happened (notebooks 06 and 11 showed agents claiming actions that never ran). The file system is the system of record.
</details>

<details><summary><b>5.</b> Which LangGraph feature did you have to build yourself here, and what are its limits?</summary>

Per-step checkpointing. Our `step()` helper only covers the steps we wrapped, saves at step granularity (not inside an agent's run), and doesn't give time travel or pausing at arbitrary points.
</details>

## Section complete 🎉

You've built a planned, parallel, self-reviewing, crash-tolerant,
human-approved research workflow on the OpenAI Agents SDK, with agents doing
the judgement and plain Python doing the orchestration.

**Where to go next:** build the same project in another framework section, then
compare the versions side by side in the comparison section.